# Coordinates: from homology classes to maps

`tda.coords` turns persistent H1 classes into coordinates on the spaces they
classify: one class → a circle, several jointly decorrelated classes → a torus.
Both come from `circular`. Every data point gets a value; classes are chosen
automatically, most persistent first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tda

rng = np.random.default_rng(1)

## Circular coordinates: one H1 class → angles

The dominant class of a noisy circle should recover the angle, up to
rotation and direction.

In [ ]:
theta = rng.uniform(0, 2 * np.pi, 400)
points = np.stack([np.cos(theta), np.sin(theta)], axis=1)
points += rng.normal(scale=0.05, size=points.shape)

res = tda.coords.circular(points)
angles = res.coordinates[:, 0]
print('class interval:', res.intervals[0], '  computed at scale:', res.scales[0])

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4))
ax0.scatter(*points.T, c=angles, cmap='hsv', s=8)
ax0.set_aspect('equal'); ax0.set_title('points colored by circular coordinate')
ax1.scatter(theta, angles, s=4)
ax1.set_xlabel('true angle'); ax1.set_ylabel('circular coordinate')
plt.tight_layout()

## Toroidal: several classes at once

Ask for more classes and each gets a column. Classes alive at a common scale
are decorrelated jointly (`res.groups` reports the blocks) — on two side-by-side
circles, each column tracks exactly one circle and stays flat on the other.

In [ ]:
shift = points + np.array([4.0, 0.0])
two = np.concatenate([points, shift])

res = tda.coords.circular(two, n_classes=2)
print('toroidal blocks:', res.groups)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for column, ax in enumerate(axes):
    ax.scatter(*two.T, c=res.coordinates[:, column], cmap='hsv', s=8)
    ax.set_aspect('equal'); ax.set_title(f'column {column}')
plt.tight_layout()

## Going faster: the quotient

Coordinates accept `quotient=True` to build them on the log(3)-interleaved
quotient cover instead of the full filtration. It is a controlled
approximation: much cheaper, with per-cluster values.

In [ ]:
fast = tda.coords.circular(points, quotient=True)

plt.figure(figsize=(4.5, 4))
plt.scatter(*points.T, c=fast.coordinates[:, 0], cmap='hsv', s=8)
plt.gca().set_aspect('equal')
plt.title('circular coordinate through the quotient')
plt.tight_layout()

## Fit once, transform new data

`circular_map` fits the same circular coordinates but keeps the model, so new
points can be mapped without recomputing persistence. Points outside the cover
(too far from the training data) come back as NaN.

In [ ]:
model = tda.coords.circular_map(points)

fresh_theta = rng.uniform(0, 2 * np.pi, 250)
fresh = np.stack([np.cos(fresh_theta), np.sin(fresh_theta)], axis=1)
fresh += rng.normal(scale=0.05, size=fresh.shape)
new_angles = model.transform(fresh)[:, 0]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 4))
ax0.scatter(*points.T, c=np.asarray(model.coordinates)[:, 0], cmap='hsv', s=8)
ax0.set_aspect('equal'); ax0.set_title('training data (fit)')
ax1.scatter(*fresh.T, c=new_angles, cmap='hsv', s=8)
ax1.set_aspect('equal'); ax1.set_title('new data (transform)')
plt.tight_layout()